In [1]:
import os
import torch
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

task = "AssociateRecall"
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
MODULE_DIR = ROOT + f"\\module\\{task}\\8len\\StateSpaceModel\\512dim\\"
IMAGE_DIR = ROOT + f"\\image\\{task}\\8len\\StateSpaceModel\\512dim\\"

epoch = 0
pattern = re.compile(r"layers\.\d+\.ssm\..*")
state_before = {}
while os.path.exists(MODULE_DIR + f"{epoch}epoch.pth"):

    state_dict = torch.load(MODULE_DIR + f"{epoch}epoch.pth", map_location="cpu")

    if state_before:
        for k, v in state_dict.items():
    
            if v.ndim == 2 and pattern.match(k):
                save_dir = k.replace(".", "\\")
                os.makedirs(IMAGE_DIR + save_dir, exist_ok=True)
                fig, axes = plt.subplots(1, 2, figsize=(10, 4))
                sns.heatmap(v, ax=axes[0], cmap='bwr')
                axes[0].set_title("Current")
                sns.heatmap(v - state_before[k], ax=axes[1], cmap='bwr')
                axes[1].set_title("Diff")
                
                fig.suptitle(f"{epoch} epoch")
                fig.savefig(f"{IMAGE_DIR}{save_dir}\\{epoch}epoch.png")
                plt.close(fig)

                
    state_before = state_dict    
    epoch += 1
        

In [2]:
for root, dirs, files in os.walk(IMAGE_DIR):
    if files and not dirs:
        
        frames = [Image.open(os.path.join(root, f)).convert("RGB") for f in files]
        save_path = os.path.join(os.path.dirname(root), f"{os.path.basename(root)}.gif")
        frames[0].save(
            save_path,
            save_all=True,
            append_images=frames[1:],
            duration=500,   # 1フレームあたりのミリ秒（500ms = 0.5秒）
            loop=0          # 無限ループ
        )